# Task 1: Exploratory Data Analysis
## Brent Oil Change Point Analysis — Birhan Energies

This notebook loads the raw Brent oil price data, cleans it, and examines its
trend, stationarity, and volatility properties ahead of the Bayesian change
point modeling in Task 2.

In [ ]:
import sys
from pathlib import Path

# Resolve the project root regardless of the notebook's working directory
# (VS Code / Jupyter sometimes runs notebooks from the project root instead of notebooks/)
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller

from scripts.data_loader import load_brent_prices, load_key_events

plt.rcParams['figure.figsize'] = (14, 5)
print("Project root:", PROJECT_ROOT)


## 1. Load & Inspect Data

In [ ]:
prices = load_brent_prices()
events = load_key_events()

print(f"Price data: {len(prices)} rows, {prices['Date'].min().date()} to {prices['Date'].max().date()}")
print(f"Events data: {len(events)} rows")
prices.head()


In [ ]:
prices.describe()


## 2. Raw Price Series — Trend Analysis

The plot below overlays vertical markers for each researched key event so we
can visually cross-check whether detected shifts line up with real-world
events.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(prices['Date'], prices['Price'], color='#1f4e79', linewidth=0.8)
for _, ev in events.iterrows():
    ax.axvline(ev['start_date'], color='crimson', alpha=0.25, linewidth=1)
ax.set_title('Brent Crude Oil Price (1987-2022) with Key Event Markers')
ax.set_xlabel('Date')
ax.set_ylabel('Price (USD/barrel)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_price_series_events.png', dpi=130)
plt.show()


**Observations:** The series shows a strong long-run upward trend with three
major regime shifts clearly visible: the 2008 financial crisis crash, the
2014-16 oversupply-driven decline, and the 2020 COVID/price-war collapse. This
non-stationarity in the mean means a single global mean/variance model is a
poor fit for the whole series — motivating the change point approach.

## 3. Stationarity Testing

In [ ]:
def adf_report(series, label):
    result = adfuller(series.dropna())
    print(f"--- ADF Test: {label} ---")
    print(f"ADF Statistic: {result[0]:.4f}")
    print(f"p-value: {result[1]:.4f}")
    for key, value in result[4].items():
        print(f"   Critical Value ({key}): {value:.4f}")
    conclusion = "Stationary" if result[1] < 0.05 else "Non-stationary"
    print(f"Conclusion (alpha=0.05): {conclusion}\n")

adf_report(prices['Price'], 'Raw Price')
adf_report(prices['log_price'], 'Log Price')
adf_report(prices['log_return'], 'Log Return')


**Expected result:** the raw price and log price series should fail to
reject the unit-root null (non-stationary), while log returns should reject
it (stationary). This confirms log returns are the right series for
volatility analysis, while log price is used as the modeled series in the
mean-shift change point model (Task 2), since price shifts should be
interpreted proportionally across different price regimes.

## 4. Log Returns & Volatility Clustering

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(prices['Date'], prices['log_return'], color='#404040', linewidth=0.4)
ax.set_title('Brent Daily Log Returns')
ax.set_xlabel('Date')
ax.set_ylabel('Log Return')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_log_returns.png', dpi=130)
plt.show()


In [ ]:
prices['rolling_vol_90'] = prices['log_return'].rolling(90).std()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(prices['Date'], prices['rolling_vol_90'], color='#8b0000', linewidth=1)
ax.set_title('90-Day Rolling Volatility of Log Returns')
ax.set_xlabel('Date')
ax.set_ylabel('Rolling Std Dev')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_rolling_volatility.png', dpi=130)
plt.show()


**Observations:** Volatility is clearly clustered rather than constant —
elevated during 2008-09, 2014-16, and reaching its historical peak in
March-April 2020 (COVID-19 demand collapse + Saudi-Russia price war). This
volatility clustering is a strong candidate for a future extension: a change
point model on variance, or a Markov-switching volatility regime model (see
Task 2 Advanced Extensions).

## 5. Summary Statistics by Period

In [ ]:
prices['period'] = (prices['Date'].dt.year // 5) * 5
summary = prices.groupby('period').agg(
    mean_price=('Price', 'mean'),
    std_price=('Price', 'std'),
    volatility=('log_return', 'std'),
    n_obs=('Price', 'size')
)
summary


## 6. Next Steps (Task 2)

1. Build a Bayesian change point model in PyMC on `log_price`, with a discrete
   uniform prior over `tau`.
2. Sample the posterior and check convergence (r_hat, trace plots).
3. Extract the posterior distribution of `tau`, convert to a calendar date.
4. Compare against `data/raw/key_events.csv` to associate detected change
   points with plausible real-world causes.
5. Quantify the impact: before/after mean price and % change, with
   appropriate correlation-vs-causation caveats (see
   `reports/task1_workflow_and_assumptions.md`).